In [1]:
import pandas as pd

In [2]:
parquet_file=r'C:\Users\playe\Documents\Trabajos\TFG-RAG-VDB\data_test\train-00000-of-00001.parquet'

In [3]:
df = pd.read_parquet(parquet_file, engine='fastparquet')
pd.read_parquet(parquet_file, engine='fastparquet')

,id,title,context,question,answers.text,answers.answer_start
0,5733be284776f41900661182,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,[Saint Bernadette Soubirous],[515]
1,5733be284776f4190066117f,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,[a copper statue of Christ],[188]
2,5733be284776f41900661180,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,[the Main Building],[279]
3,5733be284776f41900661181,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,[a Marian place of prayer and reflection],[381]
4,5733be284776f4190066117e,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,[a golden statue of the Virgin Mary],[92]
...,...,...,...,...,...,...
87594,5735d259012e2f140011a09d,Kathmandu,"Kathmandu Metropolitan City (KMC), in order to...",In what US state did Kathmandu first establish...,[Oregon],[229]
87595,5735d259012e2f140011a09e,Kathmandu,"Kathmandu Metropolitan City (KMC), in order to...",What was Yangon previously known as?,[Rangoon],[414]
87596,5735d259012e2f140011a09f,Kathmandu,"Kathmandu Metropolitan City (KMC), in order to...",With what Belorussian city does Kathmandu have...,[Minsk],[476]
87597,5735d259012e2f140011a0a0,Kathmandu,"Kathmandu Metropolitan City (KMC), in order to...",In what year did Kathmandu create its initial ...,[1975],[199]


In [4]:
print(df['context'][1])

Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.


In [9]:
# 1. Aislamos los contextos únicos y les regeneramos el doc_id 
# (Hacemos EXACTAMENTE lo mismo que hiciste el día de la inserción)
df_contextos = df.drop_duplicates(subset=['context']).copy()
df_contextos = df_contextos.reset_index(drop=True)
df_contextos['doc_id'] = ['doc_' + str(i) for i in range(len(df_contextos))]

# 2. Hacemos lo que tú sugeriste: nos quedamos solo con preguntas únicas
df_preguntas = df.drop_duplicates(subset=['question']).copy()

# 3. MAGIA DE PANDAS (Left Join): 
# Cruzamos las preguntas con la tabla de contextos usando la columna 'context'
# Ahora cada pregunta tendrá al lado su 'doc_id' correcto sin tener que ir a Postgres.
df_evaluacion = pd.merge(
    df_preguntas[['question', 'context']], # Nos quedamos con lo importante
    df_contextos[['context', 'doc_id']],   # El diccionario de traducción
    on='context', 
    how='left'
)

df_evaluacion

,question,context,doc_id
0,To whom did the Virgin Mary allegedly appear i...,"Architecturally, the school has a Catholic cha...",doc_0
1,What is in front of the Notre Dame Main Building?,"Architecturally, the school has a Catholic cha...",doc_0
2,The Basilica of the Sacred heart at Notre Dame...,"Architecturally, the school has a Catholic cha...",doc_0
3,What is the Grotto at Notre Dame?,"Architecturally, the school has a Catholic cha...",doc_0
4,What sits on top of the Main Building at Notre...,"Architecturally, the school has a Catholic cha...",doc_0
...,...,...,...
87350,In what US state did Kathmandu first establish...,"Kathmandu Metropolitan City (KMC), in order to...",doc_18890
87351,What was Yangon previously known as?,"Kathmandu Metropolitan City (KMC), in order to...",doc_18890
87352,With what Belorussian city does Kathmandu have...,"Kathmandu Metropolitan City (KMC), in order to...",doc_18890
87353,In what year did Kathmandu create its initial ...,"Kathmandu Metropolitan City (KMC), in order to...",doc_18890
